In [0]:
%sql
/*
Category Contribution to Total Sales
------------------------------------
Purpose:
    Calculate how much each product category contributes to overall sales.
    This identifies the strongest revenue-driving product groups.
*/

WITH category_sales AS (
    SELECT
        COALESCE(p.category, 'Unknown') AS category,
        SUM(f.sales_amount) AS total_sales
    FROM datawarehouseanalytics_gold.fact_sales f
    LEFT JOIN datawarehouseanalytics_gold.dim_products p
        ON f.product_key = p.product_key
    GROUP BY COALESCE(p.category, 'Unknown')
)

SELECT
    category,
    total_sales,
    SUM(total_sales) OVER () AS overall_sales,
    ROUND(total_sales * 100.0 / SUM(total_sales) OVER (), 2) AS percentage_of_total
FROM category_sales
ORDER BY total_sales DESC;

In [0]:
%sql
/*
Country Contribution to Total Sales

Purpose:
    Measure each country's share of total revenue.
    This helps compare market contribution across regions.
*/

WITH country_sales AS (
    SELECT
        COALESCE(c.country, 'Unknown') AS country,
        SUM(f.sales_amount) AS total_sales
    FROM datawarehouseanalytics_gold.fact_sales f
    LEFT JOIN datawarehouseanalytics_gold.dim_customers c
        ON f.customer_key = c.customer_key
    GROUP BY COALESCE(c.country, 'Unknown')
)

SELECT
    country,
    total_sales,
    ROUND(total_sales * 100.0 / SUM(total_sales) OVER (), 2) AS percentage_of_total
FROM country_sales
ORDER BY total_sales DESC;

In [0]:
%sql
/*
Product Line Contribution Within Category
Purpose:
    Analyze how much each product line contributes within its category.
    This shows the internal revenue mix of each product category.
*/

WITH product_line_sales AS (
    SELECT
        COALESCE(p.category, 'Unknown') AS category,
        COALESCE(p.product_line, 'Unknown') AS product_line,
        SUM(f.sales_amount) AS total_sales
    FROM datawarehouseanalytics_gold.fact_sales f
    LEFT JOIN datawarehouseanalytics_gold.dim_products p
        ON f.product_key = p.product_key
    GROUP BY 
        COALESCE(p.category, 'Unknown'),
        COALESCE(p.product_line, 'Unknown')
)

SELECT
    category,
    product_line,
    total_sales,
    ROUND(
        total_sales * 100.0 / SUM(total_sales) OVER (PARTITION BY category),
        2
    ) AS percentage_within_category
FROM product_line_sales
ORDER BY category, total_sales DESC;